# Instance-Wise Unlearning — Known/Unknown Traffic Classification

Notebook-native implementation of `plan_instance_wise.md`.

Data contract:

```text
one CSV row
-> 10,000 x [label_id, relative_time, direction, packet_size]
-> features [10,000, 3] + packet mask
-> pretrained AOL encoder + binary MLP
-> known (0) / unknown (1)
```

The notebook does not read raw PCAP files. An instance is a physical CSV row,
identified by source, split file, 1-based line number, and verified by SHA-256.

Run order:

1. Run all definition cells.
2. Edit `NOTEBOOK_COMMAND` and its argument list in the final configuration cell.
3. Set `RUN_PIPELINE = True`.
4. Run the final execution cell.


## 1. Imports, constants, records, and common helpers


In [ ]:
"""Instance-wise unlearning for VM-local processed network-flow CSV data.

The input contract matches the archived non-vm-code/train_pipeline_vm.ipynb:

    one CSV row
    -> 10,000 x [label_id, relative_time, direction, packet_size]
    -> features [10,000, 3] + packet mask
    -> pretrained AOL encoder + binary MLP (known=0, unknown=1)

The script supports four stages:

* ``validate-schema``: validate the small repository CSV fixture locally.
* ``train-base``: train and save the binary encoder+MLP checkpoint on the VM.
* ``make-manifests``: select nested AOL training instances to forget.
* ``unlearn``: compare relabel/rehearsal/L2UL methods and update scopes.

Raw PCAP files are not read by this script.  An instance is a physical CSV row
identified by a stable source/split/1-based-line ID and verified by a row hash.
"""

from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
import random
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Iterator, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset


def resolve_repo_root() -> Path:
    """Resolve project paths even when Jupyter starts in the notebook folder."""
    override = os.environ.get("BKCS_REPO_ROOT")
    start = Path(override).expanduser().resolve() if override else Path.cwd().resolve()
    candidates = [start] if override else [start, *start.parents]
    marker = Path("MLP-Classfication/vm_code/instance_wise/train_instance_wise.ipynb")
    for candidate in candidates:
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy repo root; mở notebook trong repo hoặc đặt BKCS_REPO_ROOT "
        "thành đường dẫn clone trên VM trước khi khởi động Jupyter."
    )


REPO_ROOT = resolve_repo_root()
MAX_PACKETS = 10_000
RAW_FIELDS_PER_PACKET = 4
PACKET_FEATURES = 3
ROW_WIDTH = MAX_PACKETS * RAW_FIELDS_PER_PACKET
EMBEDDING_DIM = 256
NUM_CLASSES = 2
KNOWN = 0
UNKNOWN = 1
INDEX_VERSION = "instance-wise-csv-index-v1"
FEATURE_TRANSFORM = "legacy_raw_v1"
DEFAULT_AOL_DIR = "/home/ubuntu/Documents/KF/data_processed/AOL_csv/icloud"
DEFAULT_UNKNOWN_DIR = "/home/ubuntu/Documents/KF/data_processed/icloud_100"
DEFAULT_PRETRAIN = str(REPO_ROOT / "MLP-Classfication/vm_code/weight_encoder_trained/pretrain_AOL.pth")
METHODS = (
    "relabel_only",
    "retain_rehearsal",
    "l2ul_mas",
    "l2ul_adv",
    "l2ul_adv_mas",
)
SCOPES = ("head_only", "last_encoder_block", "full_encoder_and_head")


@dataclass(frozen=True)
class CsvRecord:
    instance_id: str
    dataset_id: str
    csv_path: str
    byte_offset: int
    line_number: int
    row_sha256: str
    source: str
    original_label: str
    original_label_id: int
    binary_label: int
    split_file: str


@dataclass(frozen=True)
class PgdConfig:
    steps: int
    epsilon_time: float
    epsilon_size: float
    alpha_time: float
    alpha_size: float
    max_packet_size: float
    round_packet_size: bool


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def state_to_cpu(state: dict[str, Tensor]) -> dict[str, Tensor]:
    return {name: value.detach().cpu() for name, value in state.items()}


def torch_load(path: Path, map_location: str | torch.device = "cpu") -> dict[str, Any]:
    try:
        value = torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        value = torch.load(path, map_location=map_location)
    if not isinstance(value, dict):
        raise TypeError(f"Checkpoint phải là dictionary: {path}")
    return value


def parse_csv_values(value: str) -> tuple[str, ...]:
    return tuple(item.strip() for item in value.split(",") if item.strip())


def parse_int_values(value: str) -> tuple[int, ...]:
    result = tuple(int(item) for item in parse_csv_values(value))
    if not result:
        raise ValueError("Danh sách số nguyên không được rỗng")
    return result


## 2. CSV paths, byte-offset index, stable instance IDs, and splits


In [ ]:
def file_signature(path: Path) -> dict[str, object]:
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def dataset_signatures(aol_dir: Path, unknown_dir: Path) -> list[dict[str, object]]:
    paths = [
        root / name
        for root in (aol_dir, unknown_dir)
        for name in ("train_data.csv", "test_data.csv", "label_mapping.csv")
    ]
    return [file_signature(path) for path in paths]


def preflight_dataset_dirs(aol_dir: Path, unknown_dir: Path) -> None:
    if aol_dir.resolve() == unknown_dir.resolve():
        raise ValueError("AOL_DATASET_DIR và UNKNOWN_273_DATASET_DIR không được trùng nhau")
    for root in (aol_dir, unknown_dir):
        if not root.is_dir():
            raise FileNotFoundError(f"Không tìm thấy dataset directory: {root}")
        for name in ("train_data.csv", "test_data.csv", "label_mapping.csv"):
            path = root / name
            if not path.is_file() or path.stat().st_size == 0:
                raise FileNotFoundError(f"Thiếu hoặc rỗng: {path}")


def read_label_mapping(path: Path) -> dict[int, str]:
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["label_name", "label_id"]:
            raise ValueError(
                f"label_mapping.csv phải có header label_name,label_id; nhận {reader.fieldnames}"
            )
        mapping: dict[int, str] = {}
        for row in reader:
            label_id = int(row["label_id"])
            label_name = row["label_name"] or f"label_{label_id}"
            if label_id in mapping:
                raise ValueError(f"Trùng label_id={label_id} trong {path}")
            mapping[label_id] = label_name
    if not mapping:
        raise ValueError(f"Không có label trong {path}")
    return mapping


def make_instance_id(dataset_id: str, source: str, split_file: str, line_number: int) -> str:
    return f"{dataset_id}:{source}:{split_file}:{line_number}"


def index_one_csv(
    csv_path: Path,
    mapping: dict[int, str],
    dataset_id: str,
    source: str,
    binary_label: int,
    split_file: str,
) -> list[CsvRecord]:
    records: list[CsvRecord] = []
    with csv_path.open("rb") as handle:
        line_number = 0
        while True:
            byte_offset = handle.tell()
            raw = handle.readline()
            if not raw:
                break
            line_number += 1
            comma = raw.find(b",")
            if comma < 0:
                raise ValueError(f"Dòng không có dấu phẩy: {csv_path}:{line_number}")
            try:
                original_label_id = int(float(raw[:comma]))
            except ValueError as error:
                raise ValueError(f"Label không hợp lệ: {csv_path}:{line_number}") from error
            if original_label_id not in mapping:
                raise ValueError(
                    f"label_id={original_label_id} không có trong mapping: {csv_path}:{line_number}"
                )
            records.append(
                CsvRecord(
                    instance_id=make_instance_id(dataset_id, source, split_file, line_number),
                    dataset_id=dataset_id,
                    csv_path=str(csv_path.resolve()),
                    byte_offset=byte_offset,
                    line_number=line_number,
                    row_sha256=hashlib.sha256(raw.rstrip(b"\r\n")).hexdigest(),
                    source=source,
                    original_label=mapping[original_label_id],
                    original_label_id=original_label_id,
                    binary_label=binary_label,
                    split_file=split_file,
                )
            )
    if not records:
        raise ValueError(f"CSV không có instance: {csv_path}")
    return records


def load_or_build_index(
    aol_dir: Path,
    unknown_dir: Path,
    index_path: Path,
) -> tuple[list[CsvRecord], list[CsvRecord], list[dict[str, object]]]:
    signatures = dataset_signatures(aol_dir, unknown_dir)
    if index_path.is_file():
        cached = json.loads(index_path.read_text(encoding="utf-8"))
        if (
            cached.get("version") == INDEX_VERSION
            and cached.get("signatures") == signatures
        ):
            return (
                [CsvRecord(**row) for row in cached["source_train"]],
                [CsvRecord(**row) for row in cached["test"]],
                signatures,
            )

    aol_mapping = read_label_mapping(aol_dir / "label_mapping.csv")
    unknown_mapping = read_label_mapping(unknown_dir / "label_mapping.csv")
    source_train = index_one_csv(
        aol_dir / "train_data.csv", aol_mapping, "aol_v1", "AOL", KNOWN, "train_data.csv"
    ) + index_one_csv(
        unknown_dir / "train_data.csv",
        unknown_mapping,
        "icloud100_v1",
        "273",
        UNKNOWN,
        "train_data.csv",
    )
    test = index_one_csv(
        aol_dir / "test_data.csv", aol_mapping, "aol_v1", "AOL", KNOWN, "test_data.csv"
    ) + index_one_csv(
        unknown_dir / "test_data.csv",
        unknown_mapping,
        "icloud100_v1",
        "273",
        UNKNOWN,
        "test_data.csv",
    )
    write_json(
        index_path,
        {
            "version": INDEX_VERSION,
            "signatures": signatures,
            "source_train": [asdict(row) for row in source_train],
            "test": [asdict(row) for row in test],
        },
    )
    return source_train, test, signatures


def split_records(
    source_train: Sequence[CsvRecord],
    test: Sequence[CsvRecord],
    seed: int,
    valid_ratio: float,
) -> tuple[list[CsvRecord], list[CsvRecord], list[CsvRecord]]:
    groups: dict[tuple[str, str], list[CsvRecord]] = defaultdict(list)
    for record in source_train:
        groups[(record.source, record.original_label)].append(record)
    train: list[CsvRecord] = []
    validation: list[CsvRecord] = []
    rng = random.Random(seed)
    for key in sorted(groups):
        items = list(groups[key])
        rng.shuffle(items)
        n_val = max(1, round(len(items) * valid_ratio)) if len(items) >= 3 else 0
        validation.extend(items[:n_val])
        train.extend(items[n_val:])
    rng.shuffle(train)
    rng.shuffle(validation)
    result_test = list(test)
    if not train or not validation or not result_test:
        raise ValueError("Train/validation/test không được rỗng")
    if set(row.instance_id for row in train) & set(row.instance_id for row in validation):
        raise AssertionError("Train và validation bị giao nhau")
    return train, validation, result_test


## 3. Row decoding, tensor cache, and data loaders


In [ ]:
def cache_path(cache_dir: Path, record: CsvRecord) -> Path:
    key = hashlib.sha1(
        f"{FEATURE_TRANSFORM}:{ROW_WIDTH}:{record.instance_id}:{record.row_sha256}".encode()
    ).hexdigest()
    return cache_dir / f"{key}.pt"


def decode_csv_record(record: CsvRecord) -> tuple[Tensor, Tensor]:
    with Path(record.csv_path).open("rb") as handle:
        handle.seek(record.byte_offset)
        raw = handle.readline().rstrip(b"\r\n")
    if hashlib.sha256(raw).hexdigest() != record.row_sha256:
        raise ValueError(f"Row hash thay đổi: {record.instance_id}")
    values = np.fromstring(raw.decode("ascii"), sep=",", dtype=np.float32)
    if values.size != ROW_WIDTH:
        raise ValueError(
            f"Cần {ROW_WIDTH} values, nhận {values.size}: {record.instance_id}"
        )
    packets = values.reshape(MAX_PACKETS, RAW_FIELDS_PER_PACKET)
    labels = packets[:, 0]
    features_np = packets[:, 1:]
    real_mask = features_np[:, 2] != 0
    if not np.allclose(labels[real_mask], float(record.original_label_id)):
        raise ValueError(f"Label packet không đồng nhất: {record.instance_id}")
    padding = ~real_mask
    if padding.any() and not np.allclose(packets[padding], 0.0):
        raise ValueError(f"Padding phải là [0,0,0,0]: {record.instance_id}")
    features = torch.from_numpy(features_np.copy())
    mask = torch.from_numpy(real_mask.copy())
    return features, mask


def load_features(record: CsvRecord, feature_cache_dir: Path | None) -> tuple[Tensor, Tensor]:
    if feature_cache_dir is None:
        return decode_csv_record(record)
    feature_cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_path(feature_cache_dir, record)
    if path.is_file():
        payload = torch_load(path)
        return payload["features"], payload["mask"]
    features, mask = decode_csv_record(record)
    temporary = path.with_suffix(".tmp")
    torch.save({"features": features, "mask": mask}, temporary)
    temporary.replace(path)
    return features, mask


class CsvFlowDataset(Dataset[tuple[Tensor, Tensor, Tensor, str]]):
    def __init__(self, records: Sequence[CsvRecord], feature_cache_dir: Path | None):
        self.records = list(records)
        self.feature_cache_dir = feature_cache_dir
        self.labels = torch.tensor([row.binary_label for row in records], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str]:
        record = self.records[index]
        features, mask = load_features(record, self.feature_cache_dir)
        return features, mask, self.labels[index], record.instance_id


class AdversarialDataset(Dataset[tuple[Tensor, Tensor, Tensor, str]]):
    def __init__(self, payload: dict[str, Any]):
        self.features: Tensor = payload["features"]
        self.masks: Tensor = payload["masks"]
        self.targets: Tensor = payload["targets"]
        self.instance_ids: list[str] = list(payload["instance_ids"])

    def __len__(self) -> int:
        return self.features.shape[0]

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str]:
        return (
            self.features[index],
            self.masks[index],
            self.targets[index],
            self.instance_ids[index],
        )


def make_loader(
    records: Sequence[CsvRecord],
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    device: torch.device,
    feature_cache_dir: Path | None,
) -> DataLoader:
    return DataLoader(
        CsvFlowDataset(records, feature_cache_dir),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=device.type == "cuda",
        persistent_workers=num_workers > 0,
    )


def make_adversarial_loader(
    payload: dict[str, Any], batch_size: int, shuffle: bool, device: torch.device
) -> DataLoader:
    return DataLoader(
        AdversarialDataset(payload),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )


def move_batch(batch: Sequence[Any], device: torch.device) -> tuple[Tensor, Tensor, Tensor]:
    features, mask, labels = batch[:3]
    return (
        features.to(device, non_blocking=True),
        mask.to(device, non_blocking=True),
        labels.to(device, non_blocking=True),
    )


def endless_batches(loader: DataLoader) -> Iterator[Sequence[Any]]:
    """Repeat a loader without caching every decoded batch in host memory."""
    while True:
        yield from loader


## 4. Checkpoint-compatible encoder and binary MLP


In [ ]:
class FlowEncoder(nn.Module):
    """Checkpoint-compatible legacy raw-packet CNN encoder."""

    def __init__(self, max_packets: int = MAX_PACKETS, embedding_dim: int = EMBEDDING_DIM):
        super().__init__()
        self.max_packets = max_packets
        self.embedding_dim = embedding_dim

        def block(cin: int, cout: int, suffix: int) -> None:
            setattr(self, f"conv{suffix}", nn.Conv1d(cin, cout, 8))
            setattr(self, f"conv{suffix}_{suffix}", nn.Conv1d(cout, cout, 8))
            setattr(self, f"batch_norm{suffix}", nn.BatchNorm1d(cout))
            setattr(self, f"max_pool_{suffix}", nn.MaxPool1d(8, stride=4))
            setattr(self, f"dropout{suffix}", nn.Dropout(0.1))

        block(3, 32, 1)
        block(32, 64, 2)
        block(64, 128, 3)
        block(128, 256, 4)
        with torch.no_grad():
            flat = self._convs(torch.zeros(1, 3, max_packets)).flatten(1).shape[1]
        self.fc = nn.Linear(flat, embedding_dim)

    @staticmethod
    def _same_pad(x: Tensor) -> Tensor:
        return F.pad(x, (3, 4))

    def _one(self, x: Tensor, index: int, first: bool = False) -> Tensor:
        x = self._same_pad(x)
        x = (
            F.elu(getattr(self, f"conv{index}")(x))
            if first
            else F.relu(getattr(self, f"conv{index}")(x))
        )
        x = self._same_pad(x)
        x = getattr(self, f"conv{index}_{index}")(x)
        x = getattr(self, f"batch_norm{index}")(x)
        x = F.elu(x) if first else F.relu(x)
        x = getattr(self, f"max_pool_{index}")(self._same_pad(x))
        return getattr(self, f"dropout{index}")(x)

    def _convs(self, x: Tensor) -> Tensor:
        for index in range(1, 5):
            x = self._one(x, index, first=index == 1)
        return x

    def forward(self, features: Tensor, mask: Tensor | None = None) -> Tensor:
        if mask is not None:
            features = features * mask.unsqueeze(-1)
        return self.fc(self._convs(features.transpose(1, 2)).flatten(1))


class FlowModel(nn.Module):
    def __init__(
        self,
        hidden_dims: Sequence[int] = (512, 256, 128, 64, 32),
        dropout: float = 0.2,
    ):
        super().__init__()
        self.encoder = FlowEncoder()
        layers: list[nn.Module] = []
        width = EMBEDDING_DIM
        for next_width in hidden_dims:
            layers.extend(
                [
                    nn.Linear(width, next_width),
                    nn.LayerNorm(next_width),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            width = next_width
        layers.append(nn.Linear(width, NUM_CLASSES))
        self.classifier = nn.Sequential(*layers)
        self.hidden_dims = tuple(hidden_dims)
        self.dropout = dropout

    def forward(self, features: Tensor, mask: Tensor | None = None) -> tuple[Tensor, Tensor]:
        embedding = self.encoder(features, mask)
        return self.classifier(embedding), embedding


def load_pretrained_encoder(model: FlowModel, checkpoint_path: Path) -> dict[str, Any]:
    checkpoint = torch_load(checkpoint_path)
    config = checkpoint.get("model_config", {})
    required = {"max_packets": MAX_PACKETS, "hidden_size": 256, "embedding_size": 128}
    mismatch = {key: (config.get(key), expected) for key, expected in required.items() if config.get(key) != expected}
    if mismatch:
        raise ValueError(f"Checkpoint encoder không tương thích: {mismatch}")
    source_state = checkpoint.get("model_state_dict")
    if not isinstance(source_state, dict):
        raise ValueError(f"Thiếu model_state_dict trong {checkpoint_path}")
    encoder_state = {
        name.removeprefix("encoder."): value
        for name, value in source_state.items()
        if name.startswith("encoder.")
    }
    model.encoder.load_state_dict(encoder_state, strict=True)
    return {
        "path": str(checkpoint_path.resolve()),
        "sha256": sha256_file(checkpoint_path),
        "epoch": checkpoint.get("epoch"),
        "model_config": config,
    }


def resolve_device(device_name: str, allow_cpu: bool) -> torch.device:
    if device_name:
        requested = torch.device(device_name)
    else:
        requested = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if requested.type == "cuda" and not torch.cuda.is_available():
        if not allow_cpu:
            raise RuntimeError("CUDA không khả dụng; dùng VM GPU hoặc thêm --allow-cpu để debug")
        return torch.device("cpu")
    return requested


## 5. Evaluation, base training, threshold selection, and checkpoint loading


In [ ]:
def evaluate(
    model: FlowModel,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
) -> dict[str, object]:
    if len(loader.dataset) == 0:
        return {
            "loss": float("nan"),
            "accuracy": float("nan"),
            "known_recall": float("nan"),
            "unknown_recall": float("nan"),
            "balanced_accuracy": float("nan"),
            "unknown_threshold": threshold,
            "total": 0,
            "confusion_matrix": [[0, 0], [0, 0]],
        }
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = correct = known_total = known_ok = unknown_total = unknown_ok = 0
    loss_sum = 0.0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    with torch.no_grad():
        for batch in loader:
            features, mask, labels = move_batch(batch, device)
            logits, _ = model(features, mask)
            loss_sum += criterion(logits, labels).item() * labels.numel()
            predicted = (torch.softmax(logits, dim=1)[:, UNKNOWN] >= threshold).long()
            total += labels.numel()
            correct += (predicted == labels).sum().item()
            known = labels == KNOWN
            unknown = labels == UNKNOWN
            known_total += known.sum().item()
            unknown_total += unknown.sum().item()
            known_ok += ((predicted == labels) & known).sum().item()
            unknown_ok += ((predicted == labels) & unknown).sum().item()
            for true_label, predicted_label in zip(labels.cpu(), predicted.cpu()):
                confusion[int(true_label), int(predicted_label)] += 1
    known_recall = known_ok / known_total if known_total else float("nan")
    unknown_recall = unknown_ok / unknown_total if unknown_total else float("nan")
    valid_recalls = [value for value in (known_recall, unknown_recall) if not math.isnan(value)]
    balanced = sum(valid_recalls) / len(valid_recalls) if valid_recalls else float("nan")
    return {
        "loss": loss_sum / max(total, 1),
        "accuracy": correct / max(total, 1),
        "known_recall": known_recall,
        "unknown_recall": unknown_recall,
        "balanced_accuracy": balanced,
        "unknown_threshold": threshold,
        "total": total,
        "confusion_matrix": confusion.tolist(),
    }


def score_records(
    model: FlowModel, loader: DataLoader, device: torch.device, threshold: float
) -> list[dict[str, object]]:
    model.eval()
    rows: list[dict[str, object]] = []
    with torch.no_grad():
        for batch in loader:
            features, mask, labels = move_batch(batch, device)
            instance_ids = list(batch[3])
            logits, _ = model(features, mask)
            probability_unknown = torch.softmax(logits, dim=1)[:, UNKNOWN]
            predicted = (probability_unknown >= threshold).long()
            for instance_id, label, prediction, probability in zip(
                instance_ids, labels.cpu(), predicted.cpu(), probability_unknown.cpu()
            ):
                rows.append(
                    {
                        "instance_id": instance_id,
                        "label": int(label),
                        "prediction": int(prediction),
                        "probability_unknown": float(probability),
                    }
                )
    return rows


def forget_metrics(
    model: FlowModel, loader: DataLoader, device: torch.device, threshold: float
) -> dict[str, float]:
    rows = score_records(model, loader, device, threshold)
    if not rows:
        return {
            "forget_success_rate": float("nan"),
            "mean_probability_unknown": float("nan"),
            "count": 0.0,
        }
    return {
        "forget_success_rate": sum(row["prediction"] == UNKNOWN for row in rows) / len(rows),
        "mean_probability_unknown": sum(float(row["probability_unknown"]) for row in rows) / len(rows),
        "count": float(len(rows)),
    }


def class_weights(records: Sequence[CsvRecord], device: torch.device) -> Tensor:
    counts = torch.bincount(
        torch.tensor([record.binary_label for record in records]), minlength=NUM_CLASSES
    ).float()
    if (counts == 0).any():
        raise ValueError(f"Base train thiếu binary class: counts={counts.tolist()}")
    weights = counts.sum() / (NUM_CLASSES * counts)
    return (weights / weights.mean()).to(device)


def train_base_model(
    model: FlowModel,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    train_records: Sequence[CsvRecord],
    device: torch.device,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    freeze_encoder: bool,
) -> tuple[dict[str, Tensor], list[dict[str, float]]]:
    if freeze_encoder:
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_records, device))
    best_state = copy.deepcopy(model.state_dict())
    best_score = -float("inf")
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_encoder:
            model.encoder.eval()
        total_loss = 0.0
        total = 0
        for batch in train_loader:
            features, mask, labels = move_batch(batch, device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(features, mask)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(
                [parameter for parameter in model.parameters() if parameter.requires_grad], 5.0
            )
            optimizer.step()
            total_loss += loss.item() * labels.numel()
            total += labels.numel()
        metrics = evaluate(model, validation_loader, device, threshold=0.5)
        score = float(metrics["balanced_accuracy"])
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
        row = {
            "epoch": float(epoch),
            "train_loss": total_loss / max(total, 1),
            "validation_accuracy": float(metrics["accuracy"]),
            "validation_balanced_accuracy": score,
            "validation_known_recall": float(metrics["known_recall"]),
            "validation_unknown_recall": float(metrics["unknown_recall"]),
        }
        history.append(row)
        print(
            f"Base epoch {epoch}/{epochs}: loss={row['train_loss']:.4f} "
            f"val_bal_acc={score:.4f}"
        )
    return best_state, history


def select_threshold(
    model: FlowModel,
    validation_loader: DataLoader,
    device: torch.device,
    thresholds: Sequence[float],
) -> tuple[float, list[dict[str, float]]]:
    rows: list[dict[str, float]] = []
    for threshold in thresholds:
        metrics = evaluate(model, validation_loader, device, threshold)
        rows.append(
            {
                "threshold": float(threshold),
                "accuracy": float(metrics["accuracy"]),
                "balanced_accuracy": float(metrics["balanced_accuracy"]),
                "known_recall": float(metrics["known_recall"]),
                "unknown_recall": float(metrics["unknown_recall"]),
            }
        )
    best = max(
        rows,
        key=lambda row: (
            row["balanced_accuracy"],
            row["unknown_recall"],
            row["accuracy"],
        ),
    )
    return best["threshold"], rows


def records_by_ids(
    records: Sequence[CsvRecord], ids: Sequence[str], label: str
) -> list[CsvRecord]:
    mapping = {record.instance_id: record for record in records}
    missing = [instance_id for instance_id in ids if instance_id not in mapping]
    if missing:
        raise ValueError(f"{label} có instance không tồn tại: {missing[:5]}")
    return [mapping[instance_id] for instance_id in ids]


def split_ids_payload(
    train: Sequence[CsvRecord], validation: Sequence[CsvRecord], test: Sequence[CsvRecord]
) -> dict[str, list[str]]:
    return {
        "train": [row.instance_id for row in train],
        "validation": [row.instance_id for row in validation],
        "test": [row.instance_id for row in test],
    }


def load_base_context(
    args: argparse.Namespace,
    device: torch.device,
) -> tuple[
    FlowModel,
    dict[str, Any],
    list[CsvRecord],
    list[CsvRecord],
    list[CsvRecord],
    list[dict[str, object]],
]:
    checkpoint_path = Path(args.base_checkpoint).resolve()
    checkpoint = torch_load(checkpoint_path, map_location=device)
    model_config = checkpoint.get("model_config", {})
    model = FlowModel(
        hidden_dims=tuple(model_config.get("hidden_dims", (512, 256, 128, 64, 32))),
        dropout=float(model_config.get("dropout", 0.2)),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    aol_dir = Path(args.aol_dir).resolve()
    unknown_dir = Path(args.unknown_dir).resolve()
    preflight_dataset_dirs(aol_dir, unknown_dir)
    index_path = Path(args.index_path).resolve()
    source_train, source_test, signatures = load_or_build_index(
        aol_dir, unknown_dir, index_path
    )
    all_records = source_train + source_test
    split_ids = checkpoint.get("split_ids")
    if split_ids:
        train = records_by_ids(all_records, split_ids["train"], "base train")
        validation = records_by_ids(all_records, split_ids["validation"], "base validation")
        test = records_by_ids(all_records, split_ids["test"], "base test")
    else:
        print("Cảnh báo: base checkpoint cũ không có split_ids; tái tạo split deterministic.")
        train, validation, test = split_records(
            source_train, source_test, args.seed, args.valid_ratio
        )
    expected_signatures = checkpoint.get("dataset_signatures")
    if expected_signatures and expected_signatures != signatures:
        raise ValueError("Dataset signatures không khớp base checkpoint")
    checkpoint["checkpoint_path"] = str(checkpoint_path)
    checkpoint["checkpoint_sha256"] = sha256_file(checkpoint_path)
    return model, checkpoint, train, validation, test, signatures


## 6. Update scopes, MAS, constrained targeted PGD, and unlearning loop


In [ ]:
def set_scope(model: FlowModel, scope: str) -> list[str]:
    if scope not in SCOPES:
        raise ValueError(f"Scope không hợp lệ: {scope}")
    for parameter in model.parameters():
        parameter.requires_grad = False
    modules: dict[str, Sequence[nn.Module]] = {
        "head_only": (model.classifier,),
        "last_encoder_block": (
            model.encoder.conv4,
            model.encoder.conv4_4,
            model.encoder.batch_norm4,
            model.encoder.fc,
            model.classifier,
        ),
        "full_encoder_and_head": (model.encoder, model.classifier),
    }
    for module in modules[scope]:
        for parameter in module.parameters():
            parameter.requires_grad = True
    return [name for name, parameter in model.named_parameters() if parameter.requires_grad]


def set_unlearning_train_mode(model: FlowModel, scope: str) -> None:
    model.train()
    if scope == "head_only":
        model.encoder.eval()
    elif scope == "last_encoder_block":
        model.encoder.eval()
        for module in (
            model.encoder.conv4,
            model.encoder.conv4_4,
            model.encoder.batch_norm4,
            model.encoder.max_pool_4,
            model.encoder.dropout4,
            model.encoder.fc,
        ):
            module.train()
        model.classifier.train()


def estimate_mas_importance(
    model: FlowModel,
    forget_loader: DataLoader,
    device: torch.device,
    trainable_names: Sequence[str],
) -> dict[str, Tensor]:
    """Estimate MAS importance on Df using gradients of the output L2 norm."""
    model.eval()
    named = dict(model.named_parameters())
    parameters = [named[name] for name in trainable_names]
    importance = {name: torch.zeros_like(named[name]) for name in trainable_names}
    total = 0
    for batch in forget_loader:
        features, mask, _ = move_batch(batch, device)
        model.zero_grad(set_to_none=True)
        logits, _ = model(features, mask)
        output_norm = torch.norm(logits, p=2, dim=1).sum()
        gradients = torch.autograd.grad(output_norm, parameters, allow_unused=True)
        for name, gradient in zip(trainable_names, gradients):
            if gradient is not None:
                importance[name] += gradient.detach().abs()
        total += features.shape[0]
    if total == 0:
        raise ValueError("Df rỗng khi tính MAS")
    return {name: value / total for name, value in importance.items()}


def inverted_mas_anchors(importance: dict[str, Tensor]) -> dict[str, Tensor]:
    anchors: dict[str, Tensor] = {}
    for name, values in importance.items():
        minimum = values.min()
        span = values.max() - minimum
        if float(span) <= 1e-12:
            anchors[name] = torch.ones_like(values)
        else:
            anchors[name] = 1.0 - (values - minimum) / span
    return anchors


def parameter_anchor_loss(
    model: FlowModel,
    reference: dict[str, Tensor],
    anchors: dict[str, Tensor],
) -> Tensor:
    loss = torch.zeros((), device=next(model.parameters()).device)
    named = dict(model.named_parameters())
    for name, anchor in anchors.items():
        loss = loss + (anchor * (named[name] - reference[name]).pow(2)).sum() / 2
    return loss


def constrain_adversarial(
    candidate: Tensor,
    original: Tensor,
    mask: Tensor,
    config: PgdConfig,
) -> Tensor:
    real = mask.unsqueeze(-1)
    result = original.clone()

    time_delta = (candidate[:, :, 0] - original[:, :, 0]).clamp(
        -config.epsilon_time, config.epsilon_time
    )
    time_values = (original[:, :, 0] + time_delta).clamp_min(0.0)
    time_values = torch.cummax(time_values, dim=1).values
    time_values = torch.maximum(
        torch.minimum(time_values, original[:, :, 0] + config.epsilon_time),
        (original[:, :, 0] - config.epsilon_time).clamp_min(0.0),
    )
    # Preserve the timestamp origin of the first packet.
    time_values[:, 0] = original[:, 0, 0]

    size_delta = (candidate[:, :, 2] - original[:, :, 2]).clamp(
        -config.epsilon_size, config.epsilon_size
    )
    size_values = (original[:, :, 2] + size_delta).clamp(1.0, config.max_packet_size)
    if config.round_packet_size:
        size_values = size_values.round()

    result[:, :, 0] = torch.where(mask, time_values, torch.zeros_like(time_values))
    result[:, :, 1] = original[:, :, 1]
    result[:, :, 2] = torch.where(mask, size_values, torch.zeros_like(size_values))
    return torch.where(real, result, torch.zeros_like(result))


def generate_targeted_adversarial(
    base_model: FlowModel,
    forget_loader: DataLoader,
    device: torch.device,
    config: PgdConfig,
) -> dict[str, Any]:
    """Generate targeted unknown examples while preserving traffic constraints."""
    attack_model = copy.deepcopy(base_model).to(device).eval()
    for parameter in attack_model.parameters():
        parameter.requires_grad = False
    all_features: list[Tensor] = []
    all_masks: list[Tensor] = []
    all_targets: list[Tensor] = []
    all_ids: list[str] = []
    for batch in forget_loader:
        original, mask, _ = move_batch(batch, device)
        instance_ids = list(batch[3])
        target = torch.full(
            (original.shape[0],), UNKNOWN, dtype=torch.long, device=device
        )
        adversarial = original.detach().clone()
        for _ in range(config.steps):
            adversarial.requires_grad_(True)
            logits, _ = attack_model(adversarial, mask)
            target_loss = F.cross_entropy(logits, target)
            gradient = torch.autograd.grad(target_loss, adversarial)[0]
            updated = adversarial.detach().clone()
            updated[:, :, 0] -= config.alpha_time * gradient[:, :, 0].sign()
            updated[:, :, 2] -= config.alpha_size * gradient[:, :, 2].sign()
            adversarial = constrain_adversarial(updated, original, mask, config).detach()
        all_features.append(adversarial.cpu())
        all_masks.append(mask.cpu())
        all_targets.append(target.cpu())
        all_ids.extend(instance_ids)
    return {
        "features": torch.cat(all_features),
        "masks": torch.cat(all_masks),
        "targets": torch.cat(all_targets),
        "instance_ids": all_ids,
        "config": asdict(config),
    }


def adversarial_cache_key(
    checkpoint_sha256: str, forget_records: Sequence[CsvRecord], config: PgdConfig
) -> str:
    value = {
        "checkpoint": checkpoint_sha256,
        "instances": [record.instance_id for record in forget_records],
        "row_hashes": [record.row_sha256 for record in forget_records],
        "pgd": asdict(config),
        "feature_transform": FEATURE_TRANSFORM,
    }
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:20]


def load_or_generate_adversarial(
    base_model: FlowModel,
    forget_records: Sequence[CsvRecord],
    forget_loader: DataLoader,
    checkpoint_sha256: str,
    cache_dir: Path,
    device: torch.device,
    config: PgdConfig,
) -> tuple[dict[str, Any], Path]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"targeted_unknown_{adversarial_cache_key(checkpoint_sha256, forget_records, config)}.pt"
    if path.is_file():
        return torch_load(path), path
    payload = generate_targeted_adversarial(base_model, forget_loader, device, config)
    torch.save(payload, path)
    return payload, path


def run_unlearning_method(
    base_model: FlowModel,
    method: str,
    scope: str,
    forget_loader: DataLoader,
    retain_loader: DataLoader,
    adversarial_payload: dict[str, Any] | None,
    validation_loaders: dict[str, DataLoader],
    device: torch.device,
    threshold: float,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    lambda_retain: float,
    lambda_adv: float,
    lambda_mas: float,
    batch_size: int,
    max_steps_per_epoch: int,
) -> tuple[FlowModel, list[dict[str, object]], dict[str, Tensor] | None]:
    if method not in METHODS:
        raise ValueError(f"Method không hợp lệ: {method}")
    branch = copy.deepcopy(base_model).to(device)
    trainable_names = set_scope(branch, scope)
    reference = {
        name: parameter.detach().clone()
        for name, parameter in branch.named_parameters()
        if name in trainable_names
    }
    anchors: dict[str, Tensor] | None = None
    if method in ("l2ul_mas", "l2ul_adv_mas"):
        importance = estimate_mas_importance(branch, forget_loader, device, trainable_names)
        anchors = inverted_mas_anchors(importance)

    adversarial_loader: DataLoader | None = None
    if method in ("l2ul_adv", "l2ul_adv_mas"):
        if adversarial_payload is None:
            raise ValueError(f"{method} yêu cầu adversarial payload")
        adversarial_loader = make_adversarial_loader(
            adversarial_payload, batch_size, shuffle=True, device=device
        )

    optimizer = torch.optim.AdamW(
        [parameter for parameter in branch.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss()
    forget_cycle = endless_batches(forget_loader)
    retain_cycle = endless_batches(retain_loader)
    adversarial_cycle = endless_batches(adversarial_loader) if adversarial_loader else None
    if method == "retain_rehearsal":
        default_steps = max(len(forget_loader), len(retain_loader))
    else:
        default_steps = len(forget_loader)
    steps = min(default_steps, max_steps_per_epoch) if max_steps_per_epoch > 0 else default_steps
    history: list[dict[str, object]] = []

    for epoch in range(1, epochs + 1):
        set_unlearning_train_mode(branch, scope)
        sums = {"total": 0.0, "forget": 0.0, "retain": 0.0, "adv": 0.0, "mas": 0.0}
        for _ in range(steps):
            forget_features, forget_mask, _ = move_batch(next(forget_cycle), device)
            forget_target = torch.full(
                (forget_features.shape[0],), UNKNOWN, dtype=torch.long, device=device
            )
            optimizer.zero_grad(set_to_none=True)
            forget_logits, _ = branch(forget_features, forget_mask)
            forget_loss = criterion(forget_logits, forget_target)
            loss = forget_loss
            retain_loss = torch.zeros((), device=device)
            adv_loss = torch.zeros((), device=device)
            mas_loss = torch.zeros((), device=device)

            if method == "retain_rehearsal":
                retain_features, retain_mask, retain_labels = move_batch(
                    next(retain_cycle), device
                )
                retain_logits, _ = branch(retain_features, retain_mask)
                retain_loss = criterion(retain_logits, retain_labels)
                loss = loss + lambda_retain * retain_loss

            if adversarial_cycle is not None:
                adv_features, adv_mask, adv_target = move_batch(
                    next(adversarial_cycle), device
                )
                adv_logits, _ = branch(adv_features, adv_mask)
                adv_loss = criterion(adv_logits, adv_target)
                loss = loss + lambda_adv * adv_loss

            if anchors is not None:
                mas_loss = parameter_anchor_loss(branch, reference, anchors)
                loss = loss + lambda_mas * mas_loss

            loss.backward()
            nn.utils.clip_grad_norm_(
                [parameter for parameter in branch.parameters() if parameter.requires_grad], 5.0
            )
            optimizer.step()
            sums["total"] += loss.item()
            sums["forget"] += forget_loss.item()
            sums["retain"] += retain_loss.item()
            sums["adv"] += adv_loss.item()
            sums["mas"] += mas_loss.item()

        row: dict[str, object] = {
            "epoch": epoch,
            "steps": steps,
            **{f"loss_{key}": value / max(steps, 1) for key, value in sums.items()},
            "forget": forget_metrics(branch, forget_loader, device, threshold),
            "retain_validation": {
                name: evaluate(branch, loader, device, threshold)
                for name, loader in validation_loaders.items()
            },
        }
        history.append(row)
        print(
            f"[{method}/{scope}] epoch {epoch}/{epochs} "
            f"loss={row['loss_total']:.5f} "
            f"forget={row['forget']['forget_success_rate']:.4f}"
        )
    return branch, history, anchors


## 7. Forget manifests and retain evaluation slices


In [ ]:
def validate_forget_manifest(
    manifest_path: Path,
    train_records: Sequence[CsvRecord],
) -> tuple[dict[str, Any], list[CsvRecord]]:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    entries = manifest.get("instances")
    if not isinstance(entries, list) or not entries:
        raise ValueError("Forget manifest phải có danh sách instances không rỗng")
    train_by_id = {record.instance_id: record for record in train_records}
    ids = [entry["instance_id"] for entry in entries]
    if len(ids) != len(set(ids)):
        raise ValueError("Forget manifest có instance_id trùng")
    selected: list[CsvRecord] = []
    for entry in entries:
        instance_id = entry["instance_id"]
        if instance_id not in train_by_id:
            raise ValueError(f"Instance không thuộc base train: {instance_id}")
        record = train_by_id[instance_id]
        if record.source != "AOL" or record.binary_label != KNOWN:
            raise ValueError(f"Chỉ được quên AOL known instance: {instance_id}")
        if entry.get("row_sha256") and entry["row_sha256"] != record.row_sha256:
            raise ValueError(f"Row hash không khớp: {instance_id}")
        selected.append(record)
    return manifest, selected


def stratified_nested_order(records: Sequence[CsvRecord], seed: int) -> list[CsvRecord]:
    groups: dict[str, list[CsvRecord]] = defaultdict(list)
    for record in records:
        groups[record.original_label].append(record)
    rng = random.Random(seed)
    for values in groups.values():
        rng.shuffle(values)
    label_order = sorted(groups)
    rng.shuffle(label_order)
    ordered: list[CsvRecord] = []
    while any(groups.values()):
        for label in label_order:
            if groups[label]:
                ordered.append(groups[label].pop())
    return ordered


def build_slice_loaders(
    records: Sequence[CsvRecord],
    forget_labels: set[str],
    args: argparse.Namespace,
    device: torch.device,
    feature_cache_dir: Path,
) -> dict[str, DataLoader]:
    slices = {
        "same_original_label": [
            row for row in records if row.source == "AOL" and row.original_label in forget_labels
        ],
        "other_known": [
            row for row in records if row.source == "AOL" and row.original_label not in forget_labels
        ],
        "unknown_273": [row for row in records if row.source == "273"],
        "full_retain": list(records),
    }
    return {
        name: make_loader(
            rows,
            args.batch_size,
            False,
            args.num_workers,
            device,
            feature_cache_dir,
        )
        for name, rows in slices.items()
        if rows
    }


def evaluate_slices(
    model: FlowModel,
    loaders: dict[str, DataLoader],
    device: torch.device,
    threshold: float,
) -> dict[str, dict[str, object]]:
    return {
        name: evaluate(model, loader, device, threshold)
        for name, loader in loaders.items()
    }


## 8. Pipeline phase implementations


In [ ]:
def command_validate_schema(args: argparse.Namespace) -> None:
    root = Path(args.sample_dir).resolve()
    mapping = read_label_mapping(root / "label_mapping.csv")
    summary: dict[str, object] = {
        "sample_dir": str(root),
        "mapping_count": len(mapping),
        "label_zero": mapping.get(0),
        "files": {},
    }
    for file_name in ("train_data.csv", "test_data.csv"):
        path = root / file_name
        if not path.is_file():
            raise FileNotFoundError(path)
        inspected: list[dict[str, object]] = []
        row_count = 0
        with path.open("rb") as handle:
            while True:
                offset = handle.tell()
                raw = handle.readline()
                if not raw:
                    break
                row_count += 1
                if len(inspected) < args.rows:
                    comma = raw.find(b",")
                    label_id = int(float(raw[:comma]))
                    values = np.fromstring(raw.decode("ascii"), sep=",", dtype=np.float32)
                    if values.size != ROW_WIDTH:
                        raise ValueError(
                            f"{path}:{row_count} có {values.size}, cần {ROW_WIDTH} values"
                        )
                    packets = values.reshape(MAX_PACKETS, RAW_FIELDS_PER_PACKET)
                    real = packets[:, 3] != 0
                    inspected.append(
                        {
                            "line_number": row_count,
                            "byte_offset": offset,
                            "label_id": label_id,
                            "label_name": mapping[label_id],
                            "values": int(values.size),
                            "real_packets": int(real.sum()),
                            "padding_packets": int((~real).sum()),
                            "row_sha256": hashlib.sha256(raw.rstrip(b"\r\n")).hexdigest(),
                        }
                    )
        summary["files"][file_name] = {
            "bytes": path.stat().st_size,
            "rows": row_count,
            "inspected": inspected,
        }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


def common_runtime(args: argparse.Namespace) -> tuple[torch.device, Path, Path, Path, Path]:
    device = resolve_device(args.device, args.allow_cpu)
    aol_dir = Path(args.aol_dir).resolve()
    unknown_dir = Path(args.unknown_dir).resolve()
    preflight_dataset_dirs(aol_dir, unknown_dir)
    output_dir = Path(args.output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    feature_cache_dir = Path(args.feature_cache_dir).resolve()
    feature_cache_dir.mkdir(parents=True, exist_ok=True)
    index_path = Path(args.index_path).resolve()
    return device, aol_dir, unknown_dir, output_dir, feature_cache_dir


def command_train_base(args: argparse.Namespace) -> None:
    set_seed(args.seed)
    device, aol_dir, unknown_dir, output_dir, feature_cache_dir = common_runtime(args)
    source_train, source_test, signatures = load_or_build_index(
        aol_dir, unknown_dir, Path(args.index_path).resolve()
    )
    train, validation, test = split_records(
        source_train, source_test, args.seed, args.valid_ratio
    )
    write_json(
        output_dir / "split_manifest.json",
        {
            "seed": args.seed,
            "valid_ratio": args.valid_ratio,
            "dataset_signatures": signatures,
            "train": [asdict(row) for row in train],
            "validation": [asdict(row) for row in validation],
            "test": [asdict(row) for row in test],
        },
    )
    train_loader = make_loader(
        train, args.batch_size, True, args.num_workers, device, feature_cache_dir
    )
    validation_loader = make_loader(
        validation, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    test_loader = make_loader(
        test, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    model = FlowModel(dropout=args.dropout).to(device)
    pretrain_path = Path(args.pretrained_encoder).resolve()
    if not pretrain_path.is_file():
        raise FileNotFoundError(pretrain_path)
    pretrain_info = load_pretrained_encoder(model, pretrain_path)
    best_state, history = train_base_model(
        model,
        train_loader,
        validation_loader,
        train,
        device,
        args.epochs,
        args.learning_rate,
        args.weight_decay,
        freeze_encoder=not args.fine_tune_encoder,
    )
    model.load_state_dict(best_state)
    threshold, threshold_rows = select_threshold(
        model,
        validation_loader,
        device,
        tuple(float(value) for value in parse_csv_values(args.threshold_grid)),
    )
    test_metrics = evaluate(model, test_loader, device, threshold)
    checkpoint_path = output_dir / "base_model" / "best_model.pt"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "model_state_dict": state_to_cpu(model.state_dict()),
        "model_config": {
            "max_packets": MAX_PACKETS,
            "packet_features": PACKET_FEATURES,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dims": [512, 256, 128, 64, 32],
            "dropout": args.dropout,
            "feature_transform": FEATURE_TRANSFORM,
        },
        "best_unknown_threshold": threshold,
        "unknown_threshold": threshold,
        "split_ids": split_ids_payload(train, validation, test),
        "dataset_signatures": signatures,
        "seed": args.seed,
        "valid_ratio": args.valid_ratio,
        "pretrain": pretrain_info,
        "test_metrics": test_metrics,
    }
    torch.save(checkpoint, checkpoint_path)
    write_json(
        output_dir / "base_model" / "result.json",
        {
            "checkpoint": str(checkpoint_path),
            "counts": {
                "train": len(train),
                "validation": len(validation),
                "test": len(test),
                "train_binary": dict(Counter(row.binary_label for row in train)),
            },
            "history": history,
            "threshold_sweep": threshold_rows,
            "best_unknown_threshold": threshold,
            "test": test_metrics,
        },
    )
    print(f"Saved base checkpoint: {checkpoint_path}")
    print("Base test:", test_metrics)


def command_make_manifests(args: argparse.Namespace) -> None:
    set_seed(args.seed)
    device = resolve_device(args.device, args.allow_cpu)
    model, checkpoint, train, _, _, signatures = load_base_context(args, device)
    feature_cache_dir = Path(args.feature_cache_dir).resolve()
    aol_train = [row for row in train if row.source == "AOL"]
    loader = make_loader(
        aol_train, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    threshold = float(
        checkpoint.get("best_unknown_threshold", checkpoint.get("unknown_threshold", 0.5))
    )
    scores = {row["instance_id"]: row for row in score_records(model, loader, device, threshold)}
    eligible = [
        row for row in aol_train if scores[row.instance_id]["prediction"] == KNOWN
    ]
    ordered = stratified_nested_order(eligible, args.selection_seed)
    counts = parse_int_values(args.counts)
    if any(count <= 0 or count > len(ordered) for count in counts):
        raise ValueError(f"Forget counts phải nằm trong [1, {len(ordered)}]: {counts}")
    output_dir = Path(args.manifest_output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    for count in counts:
        selected = ordered[:count]
        manifest = {
            "version": 1,
            "selection": "nested_stratified_correctly_known",
            "selection_seed": args.selection_seed,
            "forget_count": count,
            "target_binary_label": UNKNOWN,
            "base_checkpoint": checkpoint["checkpoint_path"],
            "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
            "dataset_signatures": signatures,
            "instances": [
                {
                    "instance_id": row.instance_id,
                    "source": row.source,
                    "split_file": row.split_file,
                    "line_number": row.line_number,
                    "byte_offset": row.byte_offset,
                    "row_sha256": row.row_sha256,
                    "original_label": row.original_label,
                    "original_label_id": row.original_label_id,
                    "probability_unknown_before": scores[row.instance_id][
                        "probability_unknown"
                    ],
                }
                for row in selected
            ],
        }
        path = output_dir / f"forget_{count}.json"
        write_json(path, manifest)
        print(f"Saved {path}")


def command_unlearn(args: argparse.Namespace) -> None:
    device = resolve_device(args.device, args.allow_cpu)
    base_model, checkpoint, train, validation, test, signatures = load_base_context(
        args, device
    )
    threshold = float(
        checkpoint.get("best_unknown_threshold", checkpoint.get("unknown_threshold", 0.5))
    )
    manifest_path = Path(args.forget_manifest).resolve()
    manifest, forget_records = validate_forget_manifest(manifest_path, train)
    manifest_checkpoint_hash = manifest.get("base_checkpoint_sha256")
    if manifest_checkpoint_hash and manifest_checkpoint_hash != checkpoint["checkpoint_sha256"]:
        raise ValueError("Forget manifest được tạo từ base checkpoint khác")
    manifest_signatures = manifest.get("dataset_signatures")
    if manifest_signatures and manifest_signatures != signatures:
        raise ValueError("Forget manifest được tạo từ dataset signatures khác")

    forget_ids = {row.instance_id for row in forget_records}
    retain_train = [row for row in train if row.instance_id not in forget_ids]
    if len(forget_records) + len(retain_train) != len(train):
        raise AssertionError("Df union Dr phải bằng base train")
    forget_labels = {row.original_label for row in forget_records}
    feature_cache_dir = Path(args.feature_cache_dir).resolve()
    forget_loader = make_loader(
        forget_records,
        args.batch_size,
        True,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    forget_eval_loader = make_loader(
        forget_records,
        args.batch_size,
        False,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    retain_loader = make_loader(
        retain_train,
        args.batch_size,
        True,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    validation_loaders = build_slice_loaders(
        validation, forget_labels, args, device, feature_cache_dir
    )
    test_loaders = build_slice_loaders(test, forget_labels, args, device, feature_cache_dir)

    before = {
        "forget": forget_metrics(base_model, forget_eval_loader, device, threshold),
        "validation": evaluate_slices(base_model, validation_loaders, device, threshold),
        "test": evaluate_slices(base_model, test_loaders, device, threshold),
    }
    methods = parse_csv_values(args.methods)
    scopes = parse_csv_values(args.scopes)
    unknown_methods = set(methods) - set(METHODS)
    unknown_scopes = set(scopes) - set(SCOPES)
    if unknown_methods:
        raise ValueError(f"Method không hợp lệ: {sorted(unknown_methods)}")
    if unknown_scopes:
        raise ValueError(f"Scope không hợp lệ: {sorted(unknown_scopes)}")

    output_root = Path(args.output_dir).resolve() / "instance_unlearning" / manifest_path.stem
    output_root.mkdir(parents=True, exist_ok=True)
    pgd_config = PgdConfig(
        steps=args.pgd_steps,
        epsilon_time=args.pgd_epsilon_time,
        epsilon_size=args.pgd_epsilon_size,
        alpha_time=args.pgd_alpha_time,
        alpha_size=args.pgd_alpha_size,
        max_packet_size=args.max_packet_size,
        round_packet_size=not args.no_round_packet_size,
    )
    adversarial_payload: dict[str, Any] | None = None
    adversarial_path: Path | None = None
    if any(method in ("l2ul_adv", "l2ul_adv_mas") for method in methods):
        # A non-shuffled loader keeps IDs aligned with cached tensors.
        attack_loader = make_loader(
            forget_records,
            args.batch_size,
            False,
            args.num_workers,
            device,
            feature_cache_dir,
        )
        adversarial_payload, adversarial_path = load_or_generate_adversarial(
            base_model,
            forget_records,
            attack_loader,
            checkpoint["checkpoint_sha256"],
            output_root / "adversarial_cache",
            device,
            pgd_config,
        )

    summary: dict[str, Any] = {
        "base_checkpoint": checkpoint["checkpoint_path"],
        "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
        "forget_manifest": str(manifest_path),
        "forget_count": len(forget_records),
        "forget_instance_ids": [row.instance_id for row in forget_records],
        "forget_original_labels": sorted(forget_labels),
        "threshold": threshold,
        "before": before,
        "pgd_config": asdict(pgd_config),
        "adversarial_cache": str(adversarial_path) if adversarial_path else None,
        "experiments": [],
    }

    for method in methods:
        for scope in scopes:
            for repeat in range(args.repeats):
                optimization_seed = args.optimization_seed + repeat
                set_seed(optimization_seed)
                print(
                    f"Running method={method} scope={scope} repeat={repeat} "
                    f"seed={optimization_seed}"
                )
                branch, history, anchors = run_unlearning_method(
                    base_model,
                    method,
                    scope,
                    forget_loader,
                    retain_loader,
                    adversarial_payload,
                    validation_loaders,
                    device,
                    threshold,
                    args.unlearning_epochs,
                    args.unlearning_learning_rate,
                    args.unlearning_weight_decay,
                    args.lambda_retain,
                    args.lambda_adv,
                    args.lambda_mas,
                    args.batch_size,
                    args.max_steps_per_epoch,
                )
                after_forget = forget_metrics(
                    branch, forget_eval_loader, device, threshold
                )
                after_validation = evaluate_slices(
                    branch, validation_loaders, device, threshold
                )
                after_test = evaluate_slices(branch, test_loaders, device, threshold)
                directory = output_root / method / scope / f"repeat_{repeat}"
                directory.mkdir(parents=True, exist_ok=True)
                model_path = directory / "unlearned_model.pt"
                torch.save(
                    {
                        "model_state_dict": state_to_cpu(branch.state_dict()),
                        "model_config": checkpoint.get("model_config", {}),
                        "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
                        "forget_manifest": str(manifest_path),
                        "method": method,
                        "scope": scope,
                        "repeat": repeat,
                        "optimization_seed": optimization_seed,
                        "unknown_threshold": threshold,
                    },
                    model_path,
                )
                if anchors is not None:
                    torch.save(
                        {name: value.detach().cpu() for name, value in anchors.items()},
                        directory / "mas_anchors.pt",
                    )
                result = {
                    "method": method,
                    "scope": scope,
                    "repeat": repeat,
                    "optimization_seed": optimization_seed,
                    "model": str(model_path),
                    "before_forget": before["forget"],
                    "after_forget": after_forget,
                    "after_validation": after_validation,
                    "after_test": after_test,
                    "history": history,
                }
                write_json(directory / "result.json", result)
                summary["experiments"].append(result)
    write_json(output_root / "summary.json", summary)
    print(f"Saved unlearning summary: {output_root / 'summary.json'}")


## 9. Argument definitions shared by notebook phases


In [ ]:
def add_runtime_arguments(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--aol-dir", default=os.environ.get("AOL_DATASET_DIR", DEFAULT_AOL_DIR)
    )
    parser.add_argument(
        "--unknown-dir",
        default=os.environ.get("UNKNOWN_273_DATASET_DIR", DEFAULT_UNKNOWN_DIR),
    )
    parser.add_argument("--output-dir", default=str(REPO_ROOT / "artifacts/vm-training/instance-wise/base_run"))
    parser.add_argument("--feature-cache-dir", default=str(REPO_ROOT / "artifacts/vm-training/instance-wise/feature_cache"))
    parser.add_argument("--index-path", default=str(REPO_ROOT / "artifacts/vm-training/instance-wise/csv_index.json"))
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--valid-ratio", type=float, default=0.15)
    parser.add_argument("--device", default="cuda:0")
    parser.add_argument("--allow-cpu", action="store_true")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    subparsers = parser.add_subparsers(dest="command", required=True)

    validate = subparsers.add_parser(
        "validate-schema", help="Kiểm tra CSV fixture trên máy phát triển"
    )
    validate.add_argument(
        "--sample-dir",
        default=str(REPO_ROOT / "MLP-Classfication/csv_sample-20260921T063615Z-1-001/csv_sample/icloud"),
    )
    validate.add_argument("--rows", type=int, default=2)
    validate.set_defaults(handler=command_validate_schema)

    train = subparsers.add_parser("train-base", help="Train encoder+MLP binary base model")
    add_runtime_arguments(train)
    train.add_argument("--pretrained-encoder", default=DEFAULT_PRETRAIN)
    train.add_argument("--epochs", type=int, default=16)
    train.add_argument("--learning-rate", type=float, default=2e-4)
    train.add_argument("--weight-decay", type=float, default=1e-4)
    train.add_argument("--dropout", type=float, default=0.2)
    train.add_argument("--fine-tune-encoder", action="store_true")
    train.add_argument(
        "--threshold-grid",
        default="0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70",
    )
    train.set_defaults(handler=command_train_base)

    manifests = subparsers.add_parser(
        "make-manifests", help="Tạo nested forget manifests từ AOL base-train"
    )
    add_runtime_arguments(manifests)
    manifests.add_argument("--base-checkpoint", required=True)
    manifests.add_argument("--counts", default="1,4,10,50,100")
    manifests.add_argument("--selection-seed", type=int, default=42)
    manifests.add_argument(
        "--manifest-output-dir",
        default=str(REPO_ROOT / "artifacts/vm-training/instance-wise/forget_manifests"),
    )
    manifests.set_defaults(handler=command_make_manifests)

    unlearn = subparsers.add_parser("unlearn", help="Chạy instance-wise unlearning")
    add_runtime_arguments(unlearn)
    unlearn.add_argument("--base-checkpoint", required=True)
    unlearn.add_argument("--forget-manifest", required=True)
    unlearn.add_argument("--methods", default=",".join(METHODS))
    unlearn.add_argument("--scopes", default=",".join(SCOPES))
    unlearn.add_argument("--repeats", type=int, default=3)
    unlearn.add_argument("--optimization-seed", type=int, default=42)
    unlearn.add_argument("--unlearning-epochs", type=int, default=3)
    unlearn.add_argument("--unlearning-learning-rate", type=float, default=1e-4)
    unlearn.add_argument("--unlearning-weight-decay", type=float, default=1e-4)
    unlearn.add_argument("--lambda-retain", type=float, default=1.0)
    unlearn.add_argument("--lambda-adv", type=float, default=1.0)
    unlearn.add_argument("--lambda-mas", type=float, default=1.0)
    unlearn.add_argument("--max-steps-per-epoch", type=int, default=0)
    unlearn.add_argument("--pgd-steps", type=int, default=20)
    unlearn.add_argument("--pgd-epsilon-time", type=float, default=0.005)
    unlearn.add_argument("--pgd-epsilon-size", type=float, default=8.0)
    unlearn.add_argument("--pgd-alpha-time", type=float, default=0.0005)
    unlearn.add_argument("--pgd-alpha-size", type=float, default=1.0)
    unlearn.add_argument("--max-packet-size", type=float, default=65535.0)
    unlearn.add_argument("--no-round-packet-size", action="store_true")
    unlearn.set_defaults(handler=command_unlearn)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    args.handler(args)


## 10. Notebook phase configuration


In [ ]:
# OVERVIEW: Select exactly one safe, explicit pipeline phase.
# Keep False while reviewing/editing the notebook.
RUN_PIPELINE = False
NOTEBOOK_COMMAND = "validate-schema"

LOCAL_SAMPLE_DIR = (
    REPO_ROOT
    / "MLP-Classfication"
    / "csv_sample-20260921T063615Z-1-001"
    / "csv_sample"
    / "icloud"
)
VM_AOL_DATASET_DIR = Path("/home/ubuntu/Documents/KF/data_processed/AOL_csv/icloud")
VM_UNKNOWN_273_DATASET_DIR = Path("/home/ubuntu/Documents/KF/data_processed/icloud_100")
VM_OUTPUT_ROOT = REPO_ROOT / "artifacts/vm-training/instance-wise"
VM_FEATURE_CACHE = VM_OUTPUT_ROOT / "feature_cache"
VM_INDEX_PATH = VM_OUTPUT_ROOT / "csv_index.json"
VM_BASE_CHECKPOINT = VM_OUTPUT_ROOT / "base_run" / "base_model" / "best_model.pt"
VM_MANIFEST_DIR = VM_OUTPUT_ROOT / "forget_manifests"

NOTEBOOK_ARGUMENTS_BY_COMMAND = {
    "validate-schema": [
        "--sample-dir", str(LOCAL_SAMPLE_DIR),
        "--rows", "2",
    ],
    "train-base": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_273_DATASET_DIR),
        "--pretrained-encoder", DEFAULT_PRETRAIN,
        "--output-dir", str(VM_OUTPUT_ROOT / "base_run"),
        "--feature-cache-dir", str(VM_FEATURE_CACHE),
        "--index-path", str(VM_INDEX_PATH),
        "--epochs", "16",
        "--device", "cuda:0",
    ],
    "make-manifests": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_273_DATASET_DIR),
        "--base-checkpoint", str(VM_BASE_CHECKPOINT),
        "--counts", "1,4,10,50,100",
        "--selection-seed", "42",
        "--manifest-output-dir", str(VM_MANIFEST_DIR),
        "--output-dir", str(VM_OUTPUT_ROOT / "base_run"),
        "--feature-cache-dir", str(VM_FEATURE_CACHE),
        "--index-path", str(VM_INDEX_PATH),
        "--device", "cuda:0",
    ],
    "unlearn": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_273_DATASET_DIR),
        "--base-checkpoint", str(VM_BASE_CHECKPOINT),
        "--forget-manifest", str(VM_MANIFEST_DIR / "forget_4.json"),
        "--methods", "relabel_only,retain_rehearsal,l2ul_mas,l2ul_adv,l2ul_adv_mas",
        "--scopes", "head_only,last_encoder_block,full_encoder_and_head",
        "--repeats", "3",
        "--output-dir", str(VM_OUTPUT_ROOT / "base_run"),
        "--feature-cache-dir", str(VM_FEATURE_CACHE),
        "--index-path", str(VM_INDEX_PATH),
        "--device", "cuda:0",
    ],
}

if NOTEBOOK_COMMAND not in NOTEBOOK_ARGUMENTS_BY_COMMAND:
    raise ValueError(f"Unknown NOTEBOOK_COMMAND: {NOTEBOOK_COMMAND}")

print("Repo root:", REPO_ROOT)
print("Encoder checkpoint:", DEFAULT_PRETRAIN)
print("Selected command:", NOTEBOOK_COMMAND)
print("RUN_PIPELINE:", RUN_PIPELINE)


## 11. Execute selected phase


In [ ]:
# OVERVIEW: Execute only the explicitly selected notebook phase.
if RUN_PIPELINE:
    notebook_args = build_parser().parse_args(
        [NOTEBOOK_COMMAND, *NOTEBOOK_ARGUMENTS_BY_COMMAND[NOTEBOOK_COMMAND]]
    )
    notebook_args.handler(notebook_args)
else:
    print("Pipeline is disabled. Review the config, then set RUN_PIPELINE=True.")
